# 🔧 Applications — Prune, Quantize, Selective Finetune

Once you've discovered a circuit, CircuitKit provides three intervention types:

| Application | What it does | Use case |
|-------------|-------------|----------|
| **Pruning** | Structurally remove low-importance nodes | Reduce model size |
| **Quantization** | Circuit-guided mixed-precision | Reduce memory footprint |
| **Selective Finetuning** | Identify components for targeted PEFT/LoRA | Efficient adaptation |

All three follow the same pattern: **Discover → Apply → (Export)**.

- **Model:** `google/gemma-2-2b-it` (~2B params)
- **Task:** IOI
- **Runtime:** ~15-20 min on Colab T4

---

## Setup

In [ ]:
# ⚠️ Must be set before any CUDA context is created.
# If you've already run GPU cells in a prior notebook, restart the runtime first.
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

import warnings
warnings.filterwarnings('ignore', module='transformer_lens')
warnings.filterwarnings('ignore', module='lm_eval')
warnings.filterwarnings('ignore', message='.*pretrained.*model kwarg.*')

import gc
import torch
from circuitkit import Pipeline

print(f"CUDA: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

## Discover a Shared Circuit

All three applications operate on the same discovered circuit.

In [ ]:
MODEL = "google/gemma-2-2b-it"

pipe = Pipeline(
    MODEL,
    task="ioi",
    output_dir="./results/applications",
)

pipe.discover(
    algorithm="eap-ig",
    level="node",
    sparsity=0.3,
    n_examples=256,
    batch_size=2,
    ig_steps=5,
)

print(pipe._circuit)
print(f"Top 5: {pipe._circuit.top_nodes(5)}")

---
## Section A: Structural Pruning

Pruning removes the least-important nodes identified by circuit discovery. `pipe.prune()` creates a masked copy of the model (the original is preserved).

In [ ]:
# Prune at 30% sparsity
pipe.prune(sparsity=0.3, scope="both")
print("Pruning complete.")

### Sparsity Sweep

Compare different sparsity levels to find the best accuracy-size trade-off.

In [ ]:
import pandas as pd

sweep_rows = []
for s in [0.1, 0.2, 0.3, 0.5]:
    p = Pipeline(MODEL, task="ioi", output_dir=f"./results/applications/sweep_{int(s*100)}")
    p.discover(algorithm="eap-ig", n_examples=256, batch_size=2, ig_steps=5)
    p.prune(sparsity=s, scope="both")
    sweep_rows.append({
        "Sparsity": s,
        "Circuit Size": len(p._circuit),
        "Top Node": list(p._circuit.top_nodes(1).keys())[0] if p._circuit.scores else "—",
    })
    print(f"  Sparsity {s:.0%} done")

    # prune() loads self._model + creates self._pruned_model (two full model
    # copies on GPU). Release both before the next iteration to avoid OOM.
    p._model = None
    p._pruned_model = None
    del p
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

pd.DataFrame(sweep_rows)

### Export the Pruned Checkpoint

Save as a standard HuggingFace checkpoint directory.

In [ ]:
ckpt_path = pipe.export("./results/applications/pruned_checkpoint")
print(f"Checkpoint saved: {ckpt_path}")

# pipe._pruned_model was needed for export; release it now along with
# pipe._model before Section B loads its own model via pipe_q.
# pipe._artifact_path (a string) is still needed by Section B — it survives.
pipe._model = None
pipe._pruned_model = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

---
## Section B: Circuit-Guided Quantization

Mixed-precision quantization keeps the most important layers (by circuit score) at higher precision while aggressively quantizing the rest.

- `high_fraction=0.3` → top 30% of layers stay at full precision
- The rest are quantized to `bits` width

In [ ]:
# Fresh pipeline for quantization (avoids conflicts with pruning state)
pipe_q = Pipeline.from_artifact(
    pipe._artifact_path,
    model_name=MODEL,
    task="ioi",
    output_dir="./results/applications/quantized",
)

pipe_q.quantize(bits=4, high_fraction=0.3, backend="quanto")
print("Quantization complete.")

In [ ]:
# Export quantized checkpoint
q_ckpt = pipe_q.export("./results/applications/quantized_checkpoint", intervention="quantization")
print(f"Quantized checkpoint: {q_ckpt}")

# Release pipe_q's model before Section C loads its own.
pipe_q._model = None
pipe_q._pruned_model = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

---
## Section C: Selective Finetuning

Selective finetuning identifies the most important attention heads and MLP layers for targeted adaptation. The output is a `SelectionResult` that maps directly to PEFT/LoRA gradient masks.

- `top_fraction=0.15` → select the top 15% of components
- `scope="both"` → select from both attention and MLP

In [ ]:
# Use the same artifact from the earlier discovery
pipe_sf = Pipeline.from_artifact(
    pipe._artifact_path,
    model_name=MODEL,
    task="ioi",
    output_dir="./results/applications/finetune",
)

result = pipe_sf.selective_finetune(top_fraction=0.15, scope="both")
print(type(result))

In [ ]:
# Inspect the selection result
print(f"Attention layers selected: {len(result.attn)}")
print(f"MLP layers selected: {len(result.mlp)}")

# Show which layers were selected
print("\nAttention layers:")
for layer_key in sorted(result.attn.keys()):
    projections = list(result.attn[layer_key].keys())
    print(f"  {layer_key}: projections = {projections}")

print("\nMLP layers:")
for layer_key in sorted(result.mlp.keys()):
    val = result.mlp[layer_key]
    desc = f"{len(val)} neurons" if val is not None else "all (no mask)"
    print(f"  {layer_key}: {desc}")

### Using SelectionResult with PEFT/LoRA

The `SelectionResult` tells you *which* components to train. Integrating with a training loop looks like:

```python
# Conceptual — not a runnable training loop
for name, param in model.named_parameters():
    param.requires_grad = False  # freeze all

# Unfreeze only selected components
for layer_key, proj_dict in result.attn.items():
    layer_idx = int(layer_key.split('_')[1])
    for proj, indices in proj_dict.items():
        # Create a gradient mask for these specific indices
        ...
```

See `circuitkit.applications.selective_finetuning` for the full API.

---
## Summary

| Application | Method | Output | Next step |
|-------------|--------|--------|-----------|
| Pruning | `pipe.prune()` | Masked model | `pipe.export()` → HF checkpoint |
| Quantization | `pipe.quantize()` | Mixed-precision model | `pipe.export()` → HF checkpoint |
| Selective FT | `pipe.selective_finetune()` | `SelectionResult` | PEFT/LoRA gradient mask setup |

All three can reuse the same discovered circuit — no re-running discovery.